In [ ]:
import os
import re
from esa_snappy import ProductIO, GPF, jpy

# CARPETA ENTRADA .ZIP DESCARGADOS
input_dir = r"C:\argenis\RADAR\MOSAICO\1_IMAGENES"
# CARPETA SALIDA BANDAS PROCESADAS
output_dir = r"C:\argenis\RADAR\MOSAICO\2_PROCES"
os.makedirs(output_dir, exist_ok=True)

# TIPOS JAVA NECESARIOS
HashMap = jpy.get_type('java.util.HashMap')
Integer = jpy.get_type('java.lang.Integer')
Double = jpy.get_type('java.lang.Double')

def process_subswath(prod, subswath, output_path):
    """Procesa cada subswath IWx y guarda el resultado"""
    print(f" Procesando {subswath} → {os.path.basename(output_path)}")
    
    #1 TOPSAR-SPLIT (todos los bursts por defecto)
    params = HashMap()
    params.put('subswath', subswath)
    split = GPF.createProduct('TOPSAR-Split', params, prod)
    
    #2 CALIBRATION
    params = HashMap()
    params.put('outputSigmaBand', True)
    cal = GPF.createProduct('Calibration', params, split)
    
    #3 TOPSAR-DEBURST
    deburst = GPF.createProduct('TOPSAR-Deburst', HashMap(), cal)
    
    #4 TERRAIN CORRECTION
    params = HashMap()
    params.put('demName', 'SRTM 3Sec')
    params.put('demResamplingMethod', 'BILINEAR_INTERPOLATION')
    params.put('imgResamplingMethod', 'BILINEAR_INTERPOLATION')
    params.put('pixelSpacingInMeter', Double(20.0))
    params.put('mapProjection', 'WGS84(DD)')
    params.put('maskOutAreaWithoutElevation', True)
    
    # MUESTRA POLARIZACIONES DISPONIBLES, ESCOGE VV
    band_names = list(deburst.getBandNames())
    print("Bandas detectadas:", band_names)
    vv_band = [b for b in band_names if "VV" in b]
    if vv_band:
        params.put('sourceBands', ','.join(vv_band))
        print("Banda seleccionada:", vv_band)
    else:
        params.put('sourceBands', ','.join(band_names))
        print("No se encontró VV, procesando todas")
    tc = GPF.createProduct('Terrain-Correction', params, deburst)
    
    #5 SPECKLE FILTER
    params = HashMap()
    params.put('filter', 'Boxcar')
    params.put('filterSizeX', Integer(3))
    params.put('filterSizeY', Integer(3))
    speckle = GPF.createProduct('Speckle-Filter', params, tc)
    
    #6 LINEAR TO FROM dB
    db = GPF.createProduct('LinearToFromdB', HashMap(), speckle)
    
    #7 CONVERT DATATYPE
    params = HashMap()
    params.put('targetDataType', 'uint8')
    params.put('scaling', 'linear')
    params.put('scalingRange', '95.0')
    convert = GPF.createProduct('Convert-Datatype', params, db)
    
    #8 WRITE
    ProductIO.writeProduct(convert, output_path, 'BEAM-DIMAP')
    print(f" {subswath} guardado")

def rename_bands_in_product(product_path, new_band_name):
    """Renombra las bandas en archivos .dim, .hdr, .img y XML de metadatos BEAM-DIMAP"""
    
    # BEAM-DIMAP crea una carpeta con sufijo .data y un archivo .dim
    folder_path = product_path + ".data"
    dim_file = product_path + ".dim"
    
    if not os.path.exists(folder_path):
        print(f"  Carpeta .data no encontrada: {folder_path}")
        return
    
    if not os.path.exists(dim_file):
        print(f"  Archivo .dim no encontrado: {dim_file}")
        return
    
    # Busca el archivo .hdr que empiece con "Sigma0_IW"
    hdr_file = None
    img_file = None
    old_band_name = None
    
    for filename in os.listdir(folder_path):
        if filename.endswith('.hdr') and 'Sigma0_IW' in filename:
            hdr_file = os.path.join(folder_path, filename)
            old_band_name = filename.replace('.hdr', '')
            img_file = os.path.join(folder_path, old_band_name + '.img')
            break
    
    if not hdr_file:
        print(f"  No se encontró archivo .hdr con Sigma0_IW en: {folder_path}")
        return
    
    print(f"  Encontrado banda antigua: {old_band_name}")
    
    try:
        # 1. ACTUALIZAR EL ARCHIVO .dim (XML principal)
        with open(dim_file, 'r', encoding='utf-8') as f:
            dim_content = f.read()
        
        new_dim_content = dim_content.replace(old_band_name, new_band_name)
        
        with open(dim_file, 'w', encoding='utf-8') as f:
            f.write(new_dim_content)
        print(f"  ✓ Archivo .dim actualizado")
        
        # 2. ACTUALIZAR EL ARCHIVO .hdr
        with open(hdr_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Reemplazar en el .hdr
        new_content = re.sub(
            r'band names = \{ ' + re.escape(old_band_name) + r' \}',
            f'band names = {{ {new_band_name} }}',
            content
        )
        
        with open(hdr_file, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print(f"  ✓ Archivo .hdr actualizado")
        
        # 3. RENOMBRAR LOS ARCHIVOS .hdr E .img EN DISCO
        new_hdr_file = os.path.join(folder_path, new_band_name + '.hdr')
        new_img_file = os.path.join(folder_path, new_band_name + '.img')
        
        try:
            os.rename(hdr_file, new_hdr_file)
            print(f"  ✓ Archivo .hdr renombrado en disco: {new_band_name}.hdr")
        except Exception as e:
            print(f"  ✗ Error renombrando .hdr: {e}")
        
        if os.path.exists(img_file):
            try:
                os.rename(img_file, new_img_file)
                print(f"  ✓ Archivo .img renombrado en disco: {new_band_name}.img")
            except Exception as e:
                print(f"  ✗ Error renombrando .img: {e}")
        
        print(f"  ✓ Banda renombrada a: {new_band_name}")
            
    except Exception as e:
        print(f"  ✗ Error al procesar: {e}")

# BUCLE PARA TODOS LOS .ZIP
for fname in os.listdir(input_dir):
    if fname.lower().endswith(".zip"):
        zip_path = os.path.join(input_dir, fname)
        print(f"\n Procesando archivo: {fname}")
        prod = ProductIO.readProduct(zip_path)
        
        if prod is None:
            print("No se pudo abrir el producto, se omite.")
            continue
        
        print("Producto cargado")
        
        # NOMBRAR SALIDA en formato Sentinel-1 estándar
        parts = fname.replace(".zip", "").split("_")
        platform = parts[0]  # S1A
        pol = parts[4]  # 1SDV
        start_date = parts[5]  # 20220309T103420
        orbit = parts[7]  # 042243
        mission_id = parts[8]  # 0508EC
        unique_id = parts[9]  # 81AD
        
        # Extraer solo YYYYMMDD de la fecha
        date_only = start_date[:8]  # 20220309
        
        for iw in ["IW1", "IW2", "IW3"]:
            out_name = f"{platform}_{iw}_SLC__{pol}_{start_date}_{orbit}_{mission_id}_{unique_id}"
            out_path = os.path.join(output_dir, out_name)
            process_subswath(prod, iw, out_path)
            
            # RENOMBRAR BANDA DESPUÉS DEL PROCESAMIENTO
            new_band_name = f"Sigma0"
            print(f" Renombrando banda a: {new_band_name}")
            rename_bands_in_product(out_path, new_band_name)
        
        prod.dispose()

print("\n Procesamiento finalizado. Archivos guardados en:", output_dir)


 Procesando archivo: S1A_IW_SLC__1SDV_20180823T101751_20180823T101818_023372_028AF1_7881.zip
Producto cargado
 Procesando IW1 → S1A_IW1_SLC__1SDV_20180823T101751_023372_028AF1_7881
Bandas detectadas: ['Sigma0_IW1_VH', 'Sigma0_IW1_VV']
Banda seleccionada: ['Sigma0_IW1_VV']
